In [ ]:
!pip install -q -U pymupdf

# versões compatíveis e funcionais até 19/fev/2026, pelo menos
!pip install -q \
  langchain==1.2.0 \
  langchain-core==1.2.4 \
  langchain-text-splitters==1.0.0 \
  langchain-community==0.4.1 \
  langchain-openai==1.1.0 \
  langchain-huggingface==1.1.0 \
  pypdf==4.3.1 \
  sentence-transformers==3.2.0

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_core.prompts import PromptTemplate

In [ ]:
# faz o download do livro
# se der erro, confira se o arquivo existe na seção "Arquivos" no menu lateral
!wget https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf

# este é o nome do arquivo que deve ter sido baixado na célula anterior
arquivo = 'jk_couto_2ed.pdf'

loader = PyMuPDFLoader(f"/content/{arquivo}")
all_pages = loader.load()

print(f"Total de páginas carregadas: {len(all_pages)}")

In [ ]:
# Vamos fazer um "recorte", para usar apenas os capítulos 1 a 11,
# que estão nas páginas de 29 a 126, na numeração do PDF
pages = all_pages[28:126]
print(f"Total de páginas após o recorte: {len(pages)}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,      # Quantidade de caracteres por "chunk"
    chunk_overlap=200,    # Sobreposição de caracteres por "chunk"
    add_start_index=True, # Mapeia "chunk" no documento original
)

all_chunks = text_splitter.split_documents(pages)
print(f"Documento dividido em {len(all_chunks)} partes.")

In [ ]:
# Carrega um modelo de embedding gratuito do Hugging Face
# com bom suporte a português
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
# Cria o banco de dados vetorial (vector store) em memória
vector_store = InMemoryVectorStore(embeddings_model)

# Converte os "chunks" para vetores e os armazena no vector store
vector_store.add_documents(documents=all_chunks)

In [ ]:
query = "Qual a origem da família de Juscelino Kubitschek? Eles vieram de qual país e eram de qual etnia?"
retrieved_docs = vector_store.similarity_search(query, k=2)

print(f"Chunks recuperados: {len(retrieved_docs)}")
print("Texto do 1o chunk recuperado (mais relevante):")
print(retrieved_docs[0].page_content[:300])

In [ ]:
prompt = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template="""
Você é uma assistente de pesquisa que deve responder a pergunta informada,
de modo que sua resposta esteja bem fundamentada no contexto fornecido.
Se não for possível responder com base no contexto, diga que não achou informações
para responder.

# Contexto
{contexto}

# Pergunta
{pergunta}

# Instruções Resumidas
- Agora, responda a pergunta inicial com base no contexto informado.
- Não dê respostas com informações que não estejam no contexto.
- Não responda informações que não são relevantes para a pergunta acima.

# Resposta
"""
)

In [ ]:
# Aqui, é definido um modelo de linguagem para gerar as respostas
llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    max_new_tokens=512,
    do_sample=True,
)
# Instancia uma classe do LangChain que permite acessar modelos do HF
chat_model = ChatHuggingFace(llm=llm)

In [ ]:
def rag_workflow(pergunta_usuario, mostrar_chunks=False):
    # 1. Recuperação (busca semântica)
    docs = vector_store.similarity_search(pergunta_usuario, k=2)

    # (Extra): mostra o início dos chunks recuperados
    if mostrar_chunks:
        print("----")
        for i, doc in enumerate(docs):
            print(" => chunk", i+1, ":", doc.page_content[:80], "...")
        print("----")

    # 2. Montagem explícita do contexto - une todos os documentos em uma string
    full_context = "\n---\n".join(d.page_content for d in docs)

    # 3. Montagem explícita do prompt
    final_prompt = prompt.format(
        contexto=full_context,
        pergunta=pergunta_usuario
    )

    # 4. Chamada ao LLM
    response = chat_model.invoke(final_prompt)
    return response.content

In [ ]:
import textwrap

resposta1 = rag_workflow(
    "Qual a origem da família de Juscelino Kubitschek? Eles vieram de qual país e eram de qual etnia?",
    mostrar_chunks=True
)
print("Saída Final:\n")
print(textwrap.fill(resposta1, width=90))

In [ ]:
resposta2 = rag_workflow(
    "Quais traços de personalidade Juscelino parece ter puxado dos pais (pai e mãe)?"
)
print("Saída Final:\n")
print(textwrap.fill(resposta2, width=90))

In [ ]:
resposta3 = rag_workflow(
    "Como foi o início da vida política de Juscelino? Para qual cargo ele concorreu?"
)
print("Saída Final:")
print(textwrap.fill(resposta3, width=90))